In [1]:
import numpy as np 
from NMT import NMTModelBuilder
from tensorflow.keras.optimizers import Adam
from utils import * 

In [2]:
m = 10000
dataset, human_vocab, machine_vocab, inv_machine_vocab = load_dataset(m)

100%|██████████| 10000/10000 [00:00<00:00, 29375.76it/s]


In [3]:
print('dataset[:5] = ', dataset[:5])
print('human_vocab = ', human_vocab)
print('machine_vocab = ', machine_vocab)
print('inv_machine_vocab = ', inv_machine_vocab)

dataset[:5] =  [('9 may 1998', '1998-05-09'), ('10.11.1910/11/2019', '2019-11-10'), ('9/10/70', '1970-09-10'), ('24.10.25', '2025-10-24'), ('26.2.26', '2026-02-26')]
human_vocab =  {' ': 0, '-': 1, '.': 2, '/': 3, '0': 4, '1': 5, '2': 6, '3': 7, '4': 8, '5': 9, '6': 10, '7': 11, '8': 12, '9': 13, 'a': 14, 'b': 15, 'c': 16, 'd': 17, 'e': 18, 'f': 19, 'g': 20, 'h': 21, 'i': 22, 'j': 23, 'l': 24, 'm': 25, 'n': 26, 'o': 27, 'p': 28, 'r': 29, 's': 30, 't': 31, 'u': 32, 'v': 33, 'w': 34, 'y': 35, '<unk>': 36, '<pad>': 37}
machine_vocab =  {'-': 0, '0': 1, '1': 2, '2': 3, '3': 4, '4': 5, '5': 6, '6': 7, '7': 8, '8': 9, '9': 10}
inv_machine_vocab =  {0: '-', 1: '0', 2: '1', 3: '2', 4: '3', 5: '4', 6: '5', 7: '6', 8: '7', 9: '8', 10: '9'}


In [4]:
Tx = 30 # max length of input sentence
Ty = 10 # number of symbols in the output sentence, here we will set it to len(machine_vocab) for simplicity
X, Y, Xoh, Yoh = preprocess_data(dataset, human_vocab, machine_vocab, Tx, Ty)

print("X.shape:", X.shape) # (m, Tx) 
print("Y.shape:", Y.shape) # (m, Ty) 
print("Xoh.shape:", Xoh.shape) # (m, Tx, len(human_vocab))
print("Yoh.shape:", Yoh.shape) # (m, Ty, len(machine_vocab))

X.shape: (10000, 30)
Y.shape: (10000, 10)
Xoh.shape: (10000, 30, 38)
Yoh.shape: (10000, 10, 11)


In [5]:
print('x[0] = ', X[0]) # integer encoding of the first training example
print('y[0] = ', Y[0]) # integer encoding of the first training example (the output)
print('Xoh[0] = ', Xoh[0]) # one-hot encoding of the first training example
print('Yoh[0] = ', Yoh[0]) # one-hot encoding of the first training example (the output)

x[0] =  [13  0 25 14 35  0  5 13 13 12 37 37 37 37 37 37 37 37 37 37 37 37 37 37
 37 37 37 37 37 37]
y[0] =  [ 2 10 10  9  0  1  6  0  1 10]
Xoh[0] =  [[0. 0. 0. ... 0. 0. 0.]
 [1. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 1.]
 [0. 0. 0. ... 0. 0. 1.]
 [0. 0. 0. ... 0. 0. 1.]]
Yoh[0] =  [[0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0.]
 [1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]]


In [16]:
n_s = 64

builder = NMTModelBuilder(Tx=30, Ty=10, n_a=32, n_s=n_s, human_vocab_size=len(human_vocab), machine_vocab_size=(len(machine_vocab)))
model = builder.build_model()
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Human_Date_Input    │ (None, 30, 38)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_1     │ (None, 30, 64)    │     18,176 │ Human_Date_Input… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Initial_Hidden_Sta… │ (None, 64)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_layer_1   │ (None, 1, 64)     │      1,301 │ bidirectional_1[… │
│ (AttentionLayer)    │                   │            │ Initial_Hidden_S… │
│                     │                   │            │ bidirectional_1[… │
│                     │                   │            │ lstm_2[0][0],     │
│                     │                   │            │ bidirectional_1[… │
│                     │                   │            │ lstm_2[1][0],     │
│                     │                   │            │ bidirectional_1[… │
│                     │                   │            │ lstm_2[2][0],     │
│                     │                   │            │ bidirectional_1[… │
│                     │                   │            │ lstm_2[3][0],     │
│                     │                   │            │ bidirectional_1[… │
│                     │                   │            │ lstm_2[4][0],     │
│                     │                   │            │ bidirectional_1[… │
│                     │                   │            │ lstm_2[5][0],     │
│                     │                   │            │ bidirectional_1[… │
│                     │                   │            │ lstm_2[6][0],     │
│                     │                   │            │ bidirectional_1[… │
│                     │                   │            │ lstm_2[7][0],     │
│                     │                   │            │ bidirectional_1[… │
│                     │                   │            │ lstm_2[8][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Initial_Cell_State  │ (None, 64)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_2 (LSTM)       │ [(None, 64),      │     33,024 │ attention_layer_… │
│                     │ (None, 64),       │            │ Initial_Hidden_S… │
│                     │ (None, 64)]       │            │ Initial_Cell_Sta… │
│                     │                   │            │ attention_layer_… │
│                     │                   │            │ lstm_2[0][0],     │
│                     │                   │            │ lstm_2[0][2],     │
│                     │                   │            │ attention_layer_… │
│                     │                   │            │ lstm_2[1][0],     │
│                     │                   │            │ lstm_2[1][2],     │
│                     │                   │            │ attention_layer_… │
│                     │                   │            │ lstm_2[2][0],     │
│                     │                   │            │ lstm_2[2][2],     │
│                     │                   │            │ attention_layer_… │
│                     │                   │            │ lstm_2[3][0],     │
│                     │                   │            │ lstm_2[3][2],     │
│                     │                   │            │ attention_layer_

 Total params: 53,216 (207.88 KB)

 Trainable params: 53,216 (207.88 KB)

 Non-trainable params: 0 (0.00 B)

In [17]:
model.compile(optimizer=Adam(learning_rate= 0.005), loss='categorical_crossentropy', metrics=['accuracy'] * 10)

In [18]:
s0 = np.zeros((m, n_s))
c0 = np.zeros((m, n_s))
outputs = list(Yoh.swapaxes(0,1))
model.fit([Xoh, s0, c0], outputs, epochs=15, batch_size=100)


Epoch 1/15
100/100 ━━━━━━━━━━━━━━━━━━━━ 12s 20ms/step - dense_5_accuracy: 0.8326 - dense_5_accuracy_1: 0.8154 - dense_5_accuracy_2: 0.4592 - dense_5_accuracy_3: 0.1699 - dense_5_accuracy_4: 0.7735 - dense_5_accuracy_5: 0.3040 - dense_5_accuracy_6: 0.1215 - dense_5_accuracy_7: 0.6610 - dense_5_accuracy_8: 0.2799 - dense_5_accuracy_9: 0.1663 - dense_5_loss: 2.3555 - loss: 15.3423
Epoch 2/15
100/100 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - dense_5_accuracy: 0.9593 - dense_5_accuracy_1: 0.9573 - dense_5_accuracy_2: 0.7240 - dense_5_accuracy_3: 0.4799 - dense_5_accuracy_4: 0.9991 - dense_5_accuracy_5: 0.9015 - dense_5_accuracy_6: 0.5279 - dense_5_accuracy_7: 0.9984 - dense_5_accuracy_8: 0.5856 - dense_5_accuracy_9: 0.3932 - dense_5_loss: 1.5072 - loss: 6.4963
Epoch 3/15
100/100 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - dense_5_accuracy: 0.9674 - dense_5_accuracy_1: 0.9692 - dense_5_accuracy_2: 0.8058 - dense_5_accuracy_3: 0.7388 - dense_5_accuracy_4: 0.9994 - dense_5_accuracy_5: 0.9555 - dense_5_accur

In [19]:
EXAMPLES = ['3 May 1979', '5 April 09', '21th of August 2016', 'Tue 10 Jul 2007', 'Saturday May 9 2018', 'March 3 2001', 'March 3rd 2001', '1 March 2001']
s00 = np.zeros((1, n_s))
c00 = np.zeros((1, n_s))
for example in EXAMPLES:
    source = string_to_int(example, Tx, human_vocab)
    #print(source)
    source = np.array(list(map(lambda x: to_categorical(x, num_classes=len(human_vocab)), source))).swapaxes(0,1)
    source = np.swapaxes(source, 0, 1)
    source = np.expand_dims(source, axis=0)
    prediction = model.predict([source, s00, c00])
    prediction = np.argmax(prediction, axis = -1)
    output = [inv_machine_vocab[int(i)] for i in prediction]
    print("source:", example)
    print("output:", ''.join(output),"\n")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
source: 3 May 1979
output: 1979-05-03 

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
source: 5 April 09
output: 2009-04-05 

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
source: 21th of August 2016
output: 2016-08-21 



C:\Users\khale\AppData\Local\Temp\ipykernel_33136\2649609632.py:12: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  output = [inv_machine_vocab[int(i)] for i in prediction]


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
source: Tue 10 Jul 2007
output: 2007-06-06 

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
source: Saturday May 9 2018
output: 2018-05-09 

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
source: March 3 2001
output: 2001-03-03 

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
source: March 3rd 2001
output: 2001-03-03 

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
source: 1 March 2001
output: 2001-03-01 

